In [4]:
import os
import fitz

In [5]:
def extract_text_from_pdf(pdf_path):
    document = fitz.open(pdf_path)

    all_text = ""

    for page in document:
        all_text += page.get_text()

    return all_text

In [6]:
folders = [
    "../data/Policies",
    "../data/Controls",
    "../data/Forms",
    "../data/SOPS"
]

In [7]:
documents = {}

In [8]:
for folder in folders:

    for file in os.listdir(folder):

        if file.endswith(".pdf"):

            pdf_path = os.path.join(folder, file)

            text = extract_text_from_pdf(pdf_path)

            documents[file] = text

In [9]:
print(documents.keys())

dict_keys(['AML_Policy.pdf', 'Information_Security_Policy.pdf', 'KYC_Policy.pdf', 'Internal_Control_Library.pdf', 'Customer Onboarding Form.pdf', 'Regulatory Change Management SOP.pdf'])


In [10]:
print("Number of PDFs loaded:", len(documents))

Number of PDFs loaded: 6


In [11]:
print(documents["KYC_Policy.pdf"][:500])

 
ABC NATIONAL BANK 
Internal Policy Document 
KNOW YOUR CUSTOMER (KYC) POLICY 
 
Document Title 
Know Your Customer (KYC) Policy 
Bank Name 
ABC National Bank 
Version 
1.0 
Effective Date 
January 2025 
Document Owner 
Compliance Department 
Confidentiality 
Internal Use Only 
 
CONFIDENTIALITY: INTERNAL USE ONLY 
This document contains confidential and proprietary information of ABC National Bank. Unauthorized distribution, reproduction, or disclosure 
outside the Bank is strictly prohibited.


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [14]:
chunks = []

chunk_metadata = []

In [15]:
for filename, text in documents.items():

    document_chunks = text_splitter.split_text(text)

    for chunk in document_chunks:

        chunks.append(chunk)

        chunk_metadata.append(filename)

In [16]:
print("Total chunks:", len(chunks))

Total chunks: 66


In [17]:
print(chunk_metadata[:10])

['AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf', 'AML_Policy.pdf']


In [18]:
print(chunk_metadata[0])

print()

print(chunks[0][:300])

AML_Policy.pdf

ABC NATIONAL BANK 
Internal Policy Document 
ANTI-MONEY LAUNDERING (AML) POLICY 
 
Document Title 
Anti-Money Laundering (AML) Policy 
Bank Name 
ABC National Bank 
Version 
1.0 
Effective Date 
January 2025 
Document Owner 
Compliance Department 
Confidentiality 
Internal Use Only 
 
CONFIDENTIALIT


In [19]:
print(set(chunk_metadata))

{'Internal_Control_Library.pdf', 'Regulatory Change Management SOP.pdf', 'AML_Policy.pdf', 'Customer Onboarding Form.pdf', 'Information_Security_Policy.pdf', 'KYC_Policy.pdf'}


In [20]:
from collections import Counter

print(Counter(chunk_metadata))

Counter({'KYC_Policy.pdf': 16, 'AML_Policy.pdf': 15, 'Information_Security_Policy.pdf': 15, 'Internal_Control_Library.pdf': 10, 'Regulatory Change Management SOP.pdf': 6, 'Customer Onboarding Form.pdf': 4})


In [21]:
from sentence_transformers import SentenceTransformer

In [22]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
embeddings = model.encode(chunks)

In [25]:
print("Total embeddings:", len(embeddings))

Total embeddings: 66


In [26]:
print(embeddings[0].shape)

(384,)


In [27]:
import faiss
import numpy as np

In [28]:
embedding_matrix = np.array(embeddings).astype("float32")

In [29]:
print(embedding_matrix.shape)

(66, 384)


In [30]:
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

In [31]:
index.add(embedding_matrix)

In [32]:
print("Vectors stored:", index.ntotal)

Vectors stored: 66


In [33]:
query = """
Customer identity verification documents required before opening a new bank account.
"""

In [34]:
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

In [35]:
distances, indices = index.search(query_embedding, k=5)

In [36]:
for idx in indices[0]:
    print("="*80)
    print("Document :", chunk_metadata[idx])
    print()
    print(chunks[idx][:500])
    print()

Document : KYC_Policy.pdf

• Prevention of Money Laundering Act, 2002 (PMLA) and Rules made thereunder 
• Foreign Account Tax Compliance Act (FATCA) and Common Reporting Standard (CRS) guidelines 
• Aadhaar (Targeted Delivery of Financial and Other Subsidies, Benefits and Services) Act, 2016 
• Circulars, notifications, and directives issued by the RBI, Financial Intelligence Unit-India (FIU-IND), and SEBI, 
where applicable 
In the event of any conflict between this Policy and applicable regulatory requirements, the prov

Document : KYC_Policy.pdf

or the establishment of any business relationship. Identity Verification shall be conducted using Officially Valid 
Documents (OVDs) and shall include the following minimum steps: 
ABC National Bank – Know Your Customer (KYC) Policy 
Internal Use Only 
Version 1.0  |  Effective January 2025  |  Page 3 of 6 
1. Collection of the prospective customer's identity and address proof through valid OVDs, including Aadhaar and 
PAN 
2. Verification 

In [37]:
query = """
How should suspicious financial transactions be monitored and reported?
"""

In [38]:
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

In [39]:
distances, indices = index.search(query_embedding, k=5)

In [40]:
for idx in indices[0]:
    print("="*80)
    print("Document :", chunk_metadata[idx])
    print()
    print(chunks[idx][:500])
    print()

Document : AML_Policy.pdf

non-face-to-face onboarding 
Transaction Behaviour 
Consistent with declared profile and 
income 
Frequent large cash transactions, 
structuring, unexplained fund movement 
 
 
ABC National Bank – Anti-Money Laundering (AML) Policy 
Internal Use Only 
Version 1.0  |  Effective January 2025  |  Page 4 of 6 
6. Suspicious Transaction Monitoring 
The Bank shall maintain an automated Transaction Monitoring system to identify unusual patterns of activity inconsistent 
with a customer's declared profi

Document : AML_Policy.pdf

account reviews, and enhanced Transaction Monitoring thresholds calibrated to the customer's risk profile. 
8. Reporting Suspicious Transactions 
The Bank shall file the following statutory reports with the Financial Intelligence Unit – India in the prescribed formats 
and within the prescribed timelines: 
Report Type 
Trigger 
Filing Timeline 
Cash Transaction 
Report (CTR) 
Cash transactions of ₹10 lakh and above, or 
integrally connected

In [41]:
query = """
How should customer information be protected from cyber attacks?
"""

In [42]:
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

In [43]:
distances, indices = index.search(query_embedding, k=5)

In [44]:
for idx in indices[0]:
    print("="*80)
    print("Document :", chunk_metadata[idx])
    print()
    print(chunks[idx][:500])
    print()

Document : Information_Security_Policy.pdf

and including termination of employment, in addition to any civil or criminal liability arising under applicable law. 
12. Regulatory Compliance 
This Policy has been formulated in alignment with the RBI Cyber Security Framework for banks, the Information 
Technology Act, 2000 and rules made thereunder, guidelines issued by CERT-In, and applicable data protection 
legislation. The Bank's Information Security function shall maintain ongoing engagement with regulatory circulars and 
update this Po

Document : Information_Security_Policy.pdf

6. Access Control 
Access to information systems shall be governed by the principle of least privilege and segregation of duties. The 
following controls shall apply: 
1. Access rights shall be granted strictly on a need-to-know and role-appropriate basis, approved by the requestor's 
reporting manager and the system owner 
2. User access shall be reviewed on a quarterly basis by system owners, with immedia

In [46]:
import faiss

faiss.write_index(index, "../vector_db/bank_index.faiss")

print("FAISS saved.")

FAISS saved.


In [47]:
import pickle

with open("../vector_db/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

with open("../vector_db/chunk_metadata.pkl", "wb") as f:
    pickle.dump(chunk_metadata, f)

print("Metadata saved.")

Metadata saved.
